In [1]:
import os

In [2]:
%pwd

'c:\\Project\\CHICKEN-DISEASE-CLASSIFICATION-PROJECT\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Project\\CHICKEN-DISEASE-CLASSIFICATION-PROJECT'

entity

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    
@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path    
    


configuration manager

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks

        model_ckpt_dir = os.path.dirname(
            config.checkpoint_model_filepath
        )

        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)
        ])

        prepare_callback_config = PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(
                config.tensorboard_root_log_dir
            ),
            checkpoint_model_filepath=Path(
                config.checkpoint_model_filepath
            )
        )

        return prepare_callback_config

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        training_data = os.path.join(
            self.config.data_ingestion.unzip_path,
            "Chicken-fecal-images"
        )

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(
                prepare_base_model.updated_base_model_path
            ),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config


    

In [8]:
import time

In [9]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config=config
    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")

        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}"
        )

        return tf.keras.callbacks.TensorBoard(
            log_dir=tb_running_log_dir
        )


    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )


    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callbacks,
            self._create_ckpt_callbacks
        ]

In [10]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

training

In [ ]:
```python
import os
import tensorflow as tf
from pathlib import Path
from cnnClassifier.entity.config_entity import TrainingConfig


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=0.01
            ),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1. / 255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self, callback_list: list):

        self.model.fit(
            self.train_generator,
            validation_data=self.valid_generator,
            epochs=self.config.params_epochs,
            callbacks=callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

        print("Model saved at:", self.config.trained_model_path)
        print(
            "File exists:",
            os.path.exists(self.config.trained_model_path)
        )
```

In [12]:
try:
    config = ConfigurationManager()

    prepare_callbacks_config = config.get_prepare_callback_config()

    prepare_callbacks = PrepareCallback(
        config=prepare_callbacks_config
    )

    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()

    training = Training(
        config=training_config
    )

    training.get_base_model()
    training.train_valid_generator()

    training.train(
        callback_list=callback_list
    )

except Exception:
    import traceback
    traceback.print_exc()

Path: config\config.yaml
Content: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/itxhamzamaqsood/ml-project-image/raw/refs/heads/main/Chicken-fecal-images.7z', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_path': 'artifacts/data_ingestion'}, 'prepare_base_model': {'root_dir': 'artifacts/prepare_base_model', 'base_model_path': 'artifacts/prepare_base_model/base_model.h5', 'updated_base_model_path': 'artifacts/prepare_base_model/base_model_updatd.h5'}, 'prepare_callbacks': {'root_dir': 'artifacts/prepare_callbacks', 'tensorboard_root_log_dir': 'artifacts/prepare_callbacks/tensorboard_log_dir', 'checkpoint_model_filepath': 'artifacts/prepare_callbacks/checkpoint_dir/model.h5'}, 'training': {'root_dir': 'artifacts/training', 'trained_model_path': 'artifacts/training/model.h5'}}
Type: <class 'dict'>
[2026-10-19 12:42:49,910: INFO: common: yaml file config\config.yaml loaded sucessfully]
Path: para

Traceback (most recent call last):
  File "C:\Users\PMLS\AppData\Local\Temp\ipykernel_3748\721806478.py", line 21, in <module>
    training.train(
  File "C:\Users\PMLS\AppData\Local\Temp\ipykernel_3748\1456083941.py", line 62, in train
    self.model.fit(
  File "c:\Users\PMLS\anaconda3\envs\chicken\lib\site-packages\keras\src\utils\traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "c:\Users\PMLS\anaconda3\envs\chicken\lib\site-packages\keras\src\optimizers\base_optimizer.py", line 408, in _check_variables_are_known
    raise ValueError(
ValueError: Unknown variable: <Variable path=dense/kernel, shape=(25088, 2), dtype=float32, value=[[-0.00736337 -0.00724025]
 [-0.00637696  0.01534465]
 [-0.01246947 -0.00662238]
 ...
 [ 0.01474893 -0.00665934]
 [-0.00933491  0.01503239]
 [ 0.01164072 -0.0143743 ]]>. This optimizer can only be called for the variables it was originally built with. When working with a new set of variables, you shou

In [13]:
import os

print(os.getcwd())
print(os.path.exists("config/config.yaml"))

c:\Project\CHICKEN-DISEASE-CLASSIFICATION-PROJECT
True


In [14]:
os.chdir(r"C:\Project\CHICKEN-DISEASE-CLASSIFICATION-PROJECT")

print(os.getcwd())
print(os.path.exists(r"config\config.yaml"))


C:\Project\CHICKEN-DISEASE-CLASSIFICATION-PROJECT
True


In [15]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Eager execution:", tf.executing_eagerly())

TensorFlow: 2.21.0
NumPy: 2.2.6
Eager execution: True


In [16]:
print("TensorFlow:", tf.__version__)
print("Eager execution:", tf.executing_eagerly())
print("Functions run eagerly:", tf.config.functions_run_eagerly())

print("Model type:", type(training.model))
print("Input shape:", training.model.input_shape)
print("Output shape:", training.model.output_shape)
print("Optimizer:", training.model.optimizer)
print("Loss:", training.model.loss)
print("Run eagerly:", training.model.run_eagerly)

TensorFlow: 2.21.0
Eager execution: True
Functions run eagerly: False
Model type: <class 'keras.src.models.functional.Functional'>
Input shape: (None, 224, 224, 3)
Output shape: (None, 2)
Optimizer: <keras.src.optimizers.sgd.SGD object at 0x0000018A188549A0>
Loss: <LossFunctionWrapper(<function categorical_crossentropy at 0x0000018A7F91E560>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>
Run eagerly: True
